# Layer: Silver (Refined)
**Project:** Lean Logistics Data Pipeline  
**Business Domain:** E-commerce (Olist Dataset)

---
## 📑 Notebook Information
| Version | Date | Author | Summary of Changes |
| :--- | :--- | :--- | :--- |
| v1.0 | 2026-02-20 | Tássia Marchito | Initial ingestion from Bronze, schema enforcement, and PK constraints. |
| v1.1 | 2026-02-20 | Tássia Marchito | Added `tb_order_items` with Decimal(10,2) for financial precision. |
| v1.2 | 2026-02-20 | Tássia Marchito | Added Implemented `ts_` (timestamp) and `dt_` (date) prefixes for time-based columns. |
| v2.0 | 2026-02-20 | Tássia Marchito | Implemented `try_cast` for fault tolerance and strict data quality filtering for `tb_order_reviews`. |
| v2.1 | 2026-02-22 | Tássia Marchito | Implemented modular Silver processing with Window functions for deduplication and Unity Catalog PK RELY constraints. |

---
## 🎯 Objectives
The Silver layer represents the "Single Source of Truth". Our goal is to transform raw data into high-quality business entities.
* **Schema Enforcement:** Strict data typing using `cd_`, `ts_`, `dt_`, `vl_`, and `nm_`/`ds_` prefixes.
* **Fault Tolerance:** Usage of `try_cast` and `try_to_date` to handle malformed strings and column shifts from source APIs.
* **Data Cleansing:** Filtering corrupted rows in `tb_order_reviews` by validating `review_id` length and `review_score` range.
* **Governance:** Applying Unity Catalog constraints (PKs RELY) and removing redundant Bronze audit columns (`ts_ingestion`/`_ts_ingestion`).
* **Standardization:** Ensuring all 9 tables are correctly cataloged and deduplicated.

In [0]:
from pyspark.sql.functions import col, expr, upper, trim, current_timestamp, length, row_number
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import col, expr, upper, trim, current_timestamp, length, row_number
from pyspark.sql.window import Window

def get_col_info(col_name, config):
    """
    Corrected mapping logic to prevent naming collisions.
    """
    c = col_name.lower()
    # Explicit mapping from config
    if col_name in config["ts"]: return f"ts_{c.replace('_timestamp','').replace('_at','')}", "timestamp"
    if col_name in config["dt"]: return f"dt_{c.replace('_date','')}", "date"
    if col_name in config["vl"]: return f"vl_{c}", "decimal"
    
    # ID and Code standardization
    if any(x in c for x in ["_id", "id", "_code", "_prefix", "sequential", "lat", "lng"]):
        return f"cd_{c.replace('cd_', '')}", "string"
    
    prefix = "nm_" if any(x in c for x in ["name", "city", "state"]) else "ds_"
    return f"{prefix}{c}", "string"

# Processing Loop
for table, cfg in silver_config.items():
    source = f"cat_tm_services_bronze.db_logistics.{table}"
    target = f"cat_tm_services_silver.db_logistics.{table}"
    
    print(f"💎 Refining Silver: {table}")
    try:
        # 1. READ & PRE-FILTER (Fault Tolerance)
        df = spark.read.table(source)
        if table == "tb_order_reviews":
            df = df.filter((length(col("review_id")) == 32) & 
                           (expr("try_cast(review_score as int)").isin(1, 2, 3, 4, 5)))

        # 2. DEDUPLICATION (Using Bronze original names to avoid UNRESOLVED_COLUMN)
        # We partition by the PKs defined in the config
        window_spec = Window.partitionBy(*cfg["pk"]).orderBy(col("ts_ingestion").desc())
        df_dedup = df.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1)

        # 3. TRANSFORMATION (Mapping to new Silver schema)
        transform_exprs = []
        new_pk_list = []
        business_cols = [c for c in df.columns if c not in ["ts_ingestion", "_ts_ingestion", "_source_file", "row_num"]]
        
        for c in business_cols:
            new_name, d_type = get_col_info(c, cfg)
            if c in cfg["pk"]: new_pk_list.append(new_name)
            
            # Cast Logic
            if d_type == "timestamp": c_expr = expr(f"try_to_timestamp({c})")
            elif d_type == "date": c_expr = expr(f"try_to_date({c})")
            elif d_type == "decimal": c_expr = expr(f"try_cast({c} as decimal(10,2))")
            else: c_expr = upper(trim(col(c)))
            
            transform_exprs.append(c_expr.alias(new_name))

        # 4. FINAL SELECT & WRITE
        df_final = df_dedup.select(*transform_exprs).withColumn("ts_silver_at", current_timestamp())

        (df_final.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true").saveAsTable(target))
        
        # 5. GOVERNANCE (Constraints)
        pk_sql = ", ".join(new_pk_list)
        for c_pk in new_pk_list:
            spark.sql(f"ALTER TABLE {target} ALTER COLUMN {c_pk} SET NOT NULL")
        spark.sql(f"ALTER TABLE {target} ADD CONSTRAINT pk_{table}_slv PRIMARY KEY({pk_sql}) RELY")
        
        print(f"   ✅ Success!")

    except Exception as e:
        print(f"   ❌ Error in {table}: {e}")

In [0]:
# --- Silver Layer Validation: Integrity & Volume Check ---
import pandas as pd

validation_results = []
silver_schema = "cat_tm_services_silver.db_logistics"
bronze_schema = "cat_tm_services_bronze.db_logistics"

print(f"🔍 Validating Silver Tables in {silver_schema}...\n")

for table in silver_config.keys():
    try:
        # Count rows in Bronze (Source)
        bronze_count = spark.table(f"{bronze_schema}.{table}").count()
        
        # Count rows in Silver (Refined + Deduplicated)
        silver_df = spark.table(f"{silver_schema}.{table}")
        silver_count = silver_df.count()
        
        # Detect duplicates (difference between layers)
        duplicates_removed = bronze_count - silver_count
        
        validation_results.append({
            "Table": table,
            "Bronze Rows": bronze_count,
            "Silver Rows": silver_count,
            "Duplicates Removed": duplicates_removed,
            "Status": "✅ Verified" if silver_count > 0 else "⚠️ Empty"
        })
    except Exception as e:
        print(f"❌ Could not validate {table}: {e}")

# Displaying results as a clean table
pdf_results = pd.DataFrame(validation_results)
display(pdf_results)